# 13. Ders: Kariyer Terzisi - Özgeçmişlerden Anahtar Kelime Çıkarma (Bölüm 1)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Dünyaca ünlü şirketlerin İnsan Kaynakları (İK) departmanlarına günde on binlerce özgeçmiş (CV) gelir. Hiçbir insanın bu kadar çok dosyayı tek tek okuması mümkün değildir. Bu yüzden şirketler ATS (Aday Takip Sistemi) adı verilen yapay zeka destekli yazılımlar kullanır. Bu sistemler, başvurulan iş ilanındaki aranan özellikleri, adayın gönderdiği CV içindeki kelimelerle eşleştirir ve sadece yüksek skor alan adayları insanlara gösterir.

GCA Coding Jam projelerinden olan Kariyer Terzisi (Bulletproof) projemizde, biz de kendi mini ATS sistemimizi kodlayacağız. Uzun bir metinden (CV) veya bir iş ilanından en kritik anahtar kelimeleri çekecek ve adayın o işe ne kadar uygun olduğunu matematiksel olarak hesaplayacağız.

## Adım 2: Teknik Terimler ve Sözlük

- ATS (Applicant Tracking System): Özgeçmişleri otomatik olarak okuyup puanlayan aday takip yazılımları.
- Keyword Extraction (Anahtar Kelime Çıkarımı): Uzun bir metnin içinden sadece konuyu en iyi anlatan kritik kelimelerin (genellikle isimlerin) filtrelenmesi.
- POS Tagging (Part of Speech Tagging): Bir cümledeki kelimelerin gramer türlerini (isim, sıfat, fiil, zarf) yapay zeka ile otomatik tespit etme işlemidir.
- Set (Küme): Python'da benzersiz elemanları tutan ve iki kümenin kesişimini (ortak elemanlarını) bulmakta kullanılan matematiksel veri yapısıdır.

## Adım 3: Uzun Metinleri (İş İlanı ve CV) Tanımlamak

Önce sistemin inceleyeceği kurgusal bir iş ilanı ve kurgusal bir özgeçmiş yazalım.

In [ ]:
# Çok satırlı metinler üç tırnak (""" """) ile tanımlanır.
is_ilani = """
Yazılım mühendisliği takımımız için veri yapıları ve algoritmalar konusunda uzman, 
Python, Pandas ve Scikit-Learn araçlarına hakim, takım çalışmasına yatkın takım arkadaşı arıyoruz.
"""

aday_cv = """
Ben Ahmet. Üniversiteden yeni mezun oldum. Python dilini çok seviyorum. 
Daha önce Pandas ile çeşitli veri projeleri yaptım. Ayrıca takım çalışmasına uygun biriyim.
"""

print("Hedeflenen İş İlanı ve Adayın Özgeçmişi belleğe yüklendi.")

## Adım 4: Metinleri Temizlemek ve Tokenlara Ayırmak

8. dersimizde öğrendiğimiz NLTK kütüphanesini kullanarak bu uzun metinleri işlenebilir listelere dönüştürelim.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

# Noktalama işaretlerinden arındırılmış temiz bir liste döndüren kendi fonksiyonumuzu yazalım
def metni_temizle(metin):
    # Metni küçük harfe çevir
    kucuk_metin = metin.lower()
    # Parçala (Tokenize)
    kelimeler = word_tokenize(kucuk_metin)
    # Sadece harf ve rakam olanları al (Noktalamaları çöpe at)
    temiz_kelimeler = [kelime for kelime in kelimeler if kelime.isalnum()]
    return temiz_kelimeler

ilan_kelimeleri = metni_temizle(is_ilani)
cv_kelimeleri = metni_temizle(aday_cv)

print("İlan Kelimeleri:", ilan_kelimeleri[:5], "...")
print("CV Kelimeleri:", cv_kelimeleri[:5], "...")

## Adım 5: İsimleri ve Sıfatları Tespit Etme Neden Önemli?

Yukarıdaki listelerde 'için', 'çok', 'daha', 'ben' gibi tamamen gereksiz kelimeler var. Bizim aradığımız anahtar kelimeler genellikle gramer olarak 'İsim' (Noun) olan kelimelerdir (Örn: Python, Pandas, Algoritma, Veri). 
Yapay zekanın bu kelimelerin dil bilgisi türünü kendi başına anlaması işlemine POS (Part of Speech) Tagging denir.

In [ ]:
# İngilizce dil bilgisi etiketleyici paketini indiriyoruz
# Not: Bu araç İngilizce için mükemmel çalışır, Türkçe metinlerde başarı oranı %70-80 civarındadır.
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

print("Gramer analiz aracı başarıyla indirildi.")

## Adım 6: NLTK POS Tagger Kullanımı

Şimdi listemizdeki kelimelerin gramer etiketlerini bulalım.

In [ ]:
# pos_tag fonksiyonu kelimeleri analiz edip yanına gramer etiketini yazar
ilan_etiketleri = nltk.pos_tag(ilan_kelimeleri)

# İlk 6 kelimenin gramer analizini gösterelim
print("İlan Metninin Gramer Analizi (İlk 6 kelime):")
print(ilan_etiketleri[:6])

# Etiketlerin Anlamı (İngilizce kısaltmalar):
# NN: Noun (İsim)
# NNP: Proper Noun (Özel İsim - Örn: Python)
# VB: Verb (Fiil)

## Adım 7: Sadece Anahtar Kelimeleri (İsimleri) Filtreleme

Mademki yetenekler ve terimler her zaman 'isim' veya 'özel isim' kökenlidir, listemizden sadece 'NN' veya 'NNP' etiketine sahip olanları alalım.

In [ ]:
def anahtar_kelimeleri_bul(kelime_listesi):
    etiketli_liste = nltk.pos_tag(kelime_listesi)
    anahtar_kelimeler = []
    
    # Liste içindeki kelime ve etiket ikililerinde dönüyoruz
    for kelime, etiket in etiketli_liste:
        # Eğer etiket 'NN' (İsim) ile başlıyorsa bu değerli bir anahtar kelimedir
        if etiket.startswith("NN"):
            anahtar_kelimeler.append(kelime)
            
    return anahtar_kelimeler

ilanin_anahtarlari = anahtar_kelimeleri_bul(ilan_kelimeleri)
cvin_anahtarlari = anahtar_kelimeleri_bul(cv_kelimeleri)

print("İş İlanındaki Temel Yetenekler:", ilanin_anahtarlari)
print("Adayın CV'sindeki Temel Yetenekler:", cvin_anahtarlari)

## Adım 8: Set (Küme) Kullanarak Kesişimleri Bulma

İlandaki yetenekler ile adayın yeteneklerini matematiksel kümeler (Sets) olarak düşünüp aralarındaki ortak elemanları (Kesişim) bulacağız.

In [ ]:
# Listeleri Set (Küme) veri tipine dönüştürüyoruz (Set'ler aynı kelimeyi iki kez saymaz)
kume_ilan = set(ilanin_anahtarlari)
kume_cv = set(cvin_anahtarlari)

# intersection (kesişim) komutu her iki kümede ortak olanları getirir
ortak_yetenekler = kume_ilan.intersection(kume_cv)

print("Hem İlanda Hem CV'de Geçen Ortak Yetenekler (Kesişim):")
print(ortak_yetenekler)

## Adım 9: Eşleşme Skoru (Uyumluluk Yüzdesi) Hesaplama

Aday, ilanda istenen anahtar kelimelerin yüzde kaçını karşılıyor?

In [ ]:
aranan_yetenek_sayisi = len(kume_ilan)
adayda_olan_yetenek_sayisi = len(ortak_yetenekler)

# Yüzde hesabı: (Sahip Olunan / Aranan) * 100
uyumluluk_skoru = (adayda_olan_yetenek_sayisi / aranan_yetenek_sayisi) * 100

print(f"ARANAN TOPLAM YETENEK SAYISI: {aranan_yetenek_sayisi}")
print(f"ADAYIN KARŞILADIĞI YETENEK SAYISI: {adayda_olan_yetenek_sayisi}")
print(f"\n--- SİSTEM KARARI: Adayın İşe Uyumluluk Oranı: %{uyumluluk_skoru:.1f} ---")

if uyumluluk_skoru >= 50:
    print("KARAR: Aday mülakata davet edilecek.")
else:
    print("KARAR: Aday elendi.")

## Adım 10: Dene ve Keşfet - CV'yi Geliştirme

Adım 3'teki `aday_cv` metnine çıkıp adayın özelliklerine 'Scikit-Learn' ve 'Algoritma' kelimelerini ekleyerek hücreleri sırayla tekrar çalıştırın. Uyumluluk skorunun nasıl yükseldiğini (Adayın ATS sistemini nasıl geçtiğini) gözlemleyin.

## Adım 11: Sıra Sende Görevi - Kişisel Yetenek Eşleştirici

Problem: Yukarıdaki mantığı tek bir fonksiyon içinde toplayıp, dışarıdan parametre olarak herhangi iki metni alan ve eşleşme yüzdesini döndüren bir ATS motoru yazmanız isteniyor.

Gereksinimler:
1. `ats_motoru(ilan_metni, cv_metni)` adında bir fonksiyon oluşturun.
2. İçeride sırasıyla metinleri küçük harfe çevirme, tokenize etme ve set'e çevirme adımlarını kodlayın.
3. Kesişimi bulup `skor` değerini hesaplayın.
4. `return skor` diyerek yüzdesel değeri dışarı aktarın.
5. Dışarıda kendi hayalinizdeki iş ilanını ve kendi yeteneklerinizi yazarak fonksiyonu test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 13. Dersi tamamlayarak dünyadaki milyar dolarlık şirketlerin kullandığı Aday Takip Sistemlerinin (ATS) temel kelime çıkarım algoritmalarını başarıyla kodladınız.

Bazen kelimeleri saymak yetmez, koca bir raporu 2 cümleye sıkıştırmamız gerekir. Bir sonraki dersimizde Yapay Zeka kullanarak uzun metinleri nasıl akıllıca özetleyeceğimizi (Text Summarization) göreceğiz.